In [31]:
import torch

data = torch.Tensor([1, 2, 3])

def softmax_safe(data):
  x = data - torch.max(data)
  data_exp = torch.exp(x)
  output = data_exp / data_exp.sum(dim=0)
  return output

def softmax(data):
  data_exp = torch.exp(data)
  output = data_exp / data_exp.sum(dim=0)
  return output

class softmax_online:
  def __init__(self):
    self.max_data = float('-inf') # Initialize with negative infinity
    self.l = 0.0 # Running sum of exponentials

  def update(self, data: torch.Tensor):
    curr_max = torch.max(data)

    # Update the new max and running sum of the exponentials
    if curr_max > self.max_data:
      prev_max = self.max_data
      self.max_data = curr_max
      self.l = self.l * torch.exp(prev_max - self.max_data) + torch.sum(torch.exp(data - self.max_data))
    else:
      self.l += torch.sum(torch.exp(data - self.max_data))

  def get_softmax(self, data: torch.Tensor):
    return torch.exp(data - self.max_data) / self.l

In [32]:
# Test for online softmax
online_sm = softmax_online()
full_data = torch.Tensor([1.0, 2.0, 3.0, 4.0, 5.0])

chunk1 = full_data[:2]  # [1, 2]
chunk2 = full_data[2:3] # [3]
chunk3 = full_data[3:]  # [4, 5]

online_sm.update(chunk1)
online_sm.update(chunk2)
online_sm.update(chunk3)

online_result = online_sm.get_softmax(full_data)

standard_result = softmax_safe(full_data)

print("Online Softmax Result: ", online_result)
print("Standard Softmax Result:", standard_result)
print("Are the results equal: ", torch.allclose(online_result, standard_result))

Online Softmax Result:  tensor([0.0117, 0.0317, 0.0861, 0.2341, 0.6364])
Standard Softmax Result: tensor([0.0117, 0.0317, 0.0861, 0.2341, 0.6364])
Are the results equal:  True


In [35]:
def standard_attention(Q, K, V):
    # Q, K, V shape: [seq_len, d_model]
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / torch.sqrt(torch.tensor(d_k))
    attn_weights = torch.softmax(scores, dim=-1)
    return torch.matmul(attn_weights, V)

### What is Flash Attention?

Flash Attention is an algorithm designed to speed up the attention mechanism while using significantly less memory.

**Key Concepts:**
1. **Tiling:** It breaks the large Query, Key, and Value matrices into small blocks (tiles) that fit into the GPU's fast **SRAM** (on-chip memory). This avoids frequent, slow accesses to the main GPU memory (**HBM**).
2. **Online Softmax:** Because it processes data in chunks, it can't see the whole sequence at once to find the maximum value for softmax normalization. It uses the **online softmax** technique (tracking running maximums and sums) to compute the results incrementally and accurately.
3. **Recomputation:** Instead of storing the massive $O(N^2)$ attention matrix for the backward pass, Flash Attention recomputes it on the fly, saving a huge amount of memory at the cost of some extra computation.

In [36]:
def flash_attention_v1(Q, K, V, B=2):
    N, d = Q.shape # N - Sequence length, d - dimension of each token
    O = torch.zeros_like(Q) # Placeholder to store output
    L = torch.zeros(N) # Running sum of exponentials
    M = torch.full((N,), float('-inf')) # Running max

    # Iterate over blocks of K, V
    for j in range(0, N, B):
        # Get the tiled data by slicing across the number of the token
        Kj = K[j:j+B]
        Vj = V[j:j+B]

        # Iterate over blocks of Q
        for i in range(0, N, B):
            Qi = Q[i:i+B]

            # Calculate attention scores for the block
            S_ij = torch.matmul(Qi, Kj.transpose(-2, -1)) / torch.sqrt(torch.tensor(d))

            # Online Softmax update logic for the block
            m_ij = torch.max(S_ij, dim=-1)[0]
            p_ij = torch.exp(S_ij - m_ij.unsqueeze(-1))
            l_ij = torch.sum(p_ij, dim=-1)

            m_new = torch.max(M[i:i+B], m_ij)

            # Update scaling factors
            alpha = torch.exp(M[i:i+B] - m_new)
            beta = torch.exp(m_ij - m_new)

            L[i:i+B] = alpha * L[i:i+B] + beta * l_ij

            # Update Output weighted sum
            O[i:i+B] = (O[i:i+B] * (alpha.unsqueeze(-1))) + (beta.unsqueeze(-1) * torch.matmul(p_ij, Vj))
            M[i:i+B] = m_new

    O = O / L.unsqueeze(-1)
    return O

In [38]:
# Comparison and Validation
seq_len = 8
d_model = 16

Q = torch.randn(seq_len, d_model)
K = torch.randn(seq_len, d_model)
V = torch.randn(seq_len, d_model)

std_out = standard_attention(Q, K, V)
flash_out = flash_attention_v1(Q, K, V, B=2)

print("Standard Attention Output (first row):\n", std_out[0])
print("Flash Attention Output (first row):\n", flash_out[0])
print(f"\nAre Results equal: {torch.allclose(std_out, flash_out, atol=1e-6)}")

Standard Attention Output (first row):
 tensor([ 0.5533, -0.0061,  0.1764,  0.5593,  0.2252, -0.0477, -0.2163, -0.2140,
        -0.2071,  0.1287,  0.4627, -0.4586,  0.8209,  0.4990,  0.2715, -0.3530])
Flash Attention Output (first row):
 tensor([ 0.5533, -0.0061,  0.1764,  0.5593,  0.2252, -0.0477, -0.2163, -0.2140,
        -0.2071,  0.1287,  0.4627, -0.4586,  0.8209,  0.4990,  0.2715, -0.3530])

Are Results equal: True


In [39]:
import timeit

# Increase size for more meaningful comparison
seq_len_bench = 128
d_model_bench = 64

Q_b = torch.randn(seq_len_bench, d_model_bench)
K_b = torch.randn(seq_len_bench, d_model_bench)
V_b = torch.randn(seq_len_bench, d_model_bench)

print(f"Benchmarking for sequence length {seq_len_bench}...\n")

print("Standard Attention:")
%timeit standard_attention(Q_b, K_b, V_b)

print("\nFlash Attention (Scratch Python Implementation):")
# Note: This implementation is in Python, so it will be slower than the optimized
# torch.softmax due to interpreter overhead, even though the algorithm is more IO-efficient.
%timeit flash_attention_v1(Q_b, K_b, V_b, B=32)

Benchmarking for sequence length 128...

Standard Attention:
150 µs ± 29.2 µs per loop (mean ± std. dev. of 7 runs, 10000 loops each)

Flash Attention (Scratch Python Implementation):
2.73 ms ± 429 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


### Note on Speed
`standard_attention` is faster in this implementation because:
1.  **Standard Attention** uses highly optimized C++/CUDA kernels for `torch.softmax` and `matmul`.
2.  **Our Flash Attention** uses Python `for` loops, which are very slow in comparison to native operations.

In a real-world scenario (C++/CUDA), Flash Attention is faster because it reduces memory reads/writes to the GPU's High Bandwidth Memory (HBM) by keeping data in the fast SRAM cache, even if it does more compute.